In [44]:
import os
DEEPSEEK_API_KEY="sk-c446a31711e24b53bcdd9029b78e31e1"

In [45]:
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START

deepseek_agent = ChatDeepSeek(
    model="deepseek-chat",
    api_key="sk-c446a31711e24b53bcdd9029b78e31e1",
    temperature=0,  
    max_retries=3,  
    timeout=30      
)


In [46]:
# 导入类型注解工具
from typing import TypedDict, Annotated
from typing_extensions import TypedDict

# 定义状态类型
class AgentState(TypedDict):
    messages: list[str]

# 初始化状态机
workflow = StateGraph(AgentState)

# 定义Agent节点处理逻辑
def agent_node(state):
    response = deepseek_agent.invoke(state["messages"])
    return {"messages": [response]}

# 注册节点到状态机
workflow.add_node("deepseek_agent", agent_node)


In [47]:
from langgraph.graph import StateGraph, END
workflow.set_entry_point("deepseek_agent")
workflow.add_edge("deepseek_agent", END)
app = workflow.compile()

In [48]:
from langchain_core.messages import HumanMessage
initial_state = {"messages": [HumanMessage(content="生成Python排序算法单元测试")]}
result = app.invoke(initial_state)

APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# 创建合法PromptTemplate对象（自动推断输入变量）
prompt_template = PromptTemplate.from_template(
    "Your template content with {input}"  # 模板声明占位符
)

# 构建符合LCEL规范的执行链
chain = (
    RunnablePassthrough.assign(input=lambda x: x["input"])  # 输入预处理
    | prompt_template  # 使用自动推断变量的模板
    | deepseek_agent   # 后续处理组件
    | output_parser
)

NameError: name 'output_parser' is not defined